# Практика 3: Полная реализация трансформера

Соберём полный трансформер, включая энкодер и декодер. Начнём с реализации блока энкодера: создаём `nn.ModuleList` из нескольких слоёв внимания и `полносвязных слоёв FeedForward`, включающих два линейных преобразования и `активацию ReLU`. Добавляем `позиционные эмбеддинги` (т.к. трансформеры не обрабатывают последовательность напрямую, необходимо добавить информацию о позиции слов). Объединяем слои в полноценную сеть, `нормализуем с LayerNorm`. Далее переходим к декодеру: он похож на энкодер, но дополнительно использует механизм маскированного внимания для предсказания следующего токена.  `Энкодер - это BERT, Декодер - это GPT`. Применяем `оптимизатор AdamW`, используем `кросс-энтропийную функцию потерь`. Тестируем модель, проверяем логики её работы: подаём входной текст и анализируем `выходное распределение вероятностей по токенам`. Экспериментируем с различными настройками гиперпараметров (количество слоёв, размер скрытых представлений, число голов внимания) и анализируем влияние на качество.

In [2]:
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from tqdm.auto import tqdm

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

device

c:\Users\semen\works\NN2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
torch.cuda.empty_cache()

In [4]:
def make_causal_mask(T: int, device: torch.device) -> torch.Tensor:
    m = torch.tril(torch.ones(T, T, device=device))
    return m.unsqueeze(0).unsqueeze(0)

In [5]:
def make_padding_mask(tokens: torch.Tensor, pad_id: int) -> torch.Tensor:
    return (tokens != pad_id).unsqueeze(1).unsqueeze(2).float()

In [6]:
def scaled_dot_product_attention(Q, K, V, attn_mask=None, dropout_p=0.0, training=True):
    _, _, _, D = Q.shape
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(D)

    if attn_mask is not None:
        scores = scores.masked_fill(attn_mask == 0, float("-inf"))

    attn = torch.softmax(scores, dim=-1)

    if dropout_p > 0:
        attn = F.dropout(attn, p=dropout_p, training=training)

    out = torch.matmul(attn, V)
    return out, attn

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout_p: float = 0.0, bias: bool = True):
        super().__init__()
 
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout_p = dropout_p

        self.Wq = nn.Linear(d_model, d_model, bias=bias)
        self.Wk = nn.Linear(d_model, d_model, bias=bias)
        self.Wv = nn.Linear(d_model, d_model, bias=bias)
        self.Wo = nn.Linear(d_model, d_model, bias=bias)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, H, T, D = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * D)

    def forward(self, x_q, x_kv=None, attn_mask=None, need_weights=False):
        if x_kv is None:
            x_kv = x_q

        Q = self._split_heads(self.Wq(x_q))
        K = self._split_heads(self.Wk(x_kv))
        V = self._split_heads(self.Wv(x_kv))

        out, attn = scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p,
            training=self.training
        )
        out = self.Wo(self._merge_heads(out))

        if need_weights:
            return out, attn
        return out

In [8]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout_p: float = 0.0):
        super().__init__()
        self.lin1 = nn.Linear(d_model, d_ff)
        self.lin2 = nn.Linear(d_ff, d_model)
        self.dropout_p = dropout_p

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.lin2(x)
        return x

In [9]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        return x + self.pe[:T, :].unsqueeze(0)

In [10]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_p: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.ffn = FeedForward(d_model, d_ff, dropout_p=dropout_p)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout_p = dropout_p

    def forward(self, x, src_key_padding_mask=None):
        attn_out = self.self_attn(x, attn_mask=src_key_padding_mask)
        x = x + F.dropout(attn_out, p=self.dropout_p, training=self.training)
        x = self.norm1(x)

        ffn_out = self.ffn(x)
        x = x + F.dropout(ffn_out, p=self.dropout_p, training=self.training)
        x = self.norm2(x)
        return x

In [11]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_heads: int, d_ff: int,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.dropout_p = dropout_p

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout_p=dropout_p)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src_tokens, src_key_padding_mask=None):
        x = self.tok_emb(src_tokens)
        x = self.pos_emb(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        return self.norm(x)

In [12]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_p: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.ffn = FeedForward(d_model, d_ff, dropout_p=dropout_p)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout_p = dropout_p

    def forward(self, x, memory, tgt_attn_mask=None, src_key_padding_mask=None):
        self_out = self.self_attn(x, attn_mask=tgt_attn_mask)
        x = x + F.dropout(self_out, p=self.dropout_p, training=self.training)
        x = self.norm1(x)

        cross_out = self.cross_attn(x_q=x, x_kv=memory, attn_mask=src_key_padding_mask)
        x = x + F.dropout(cross_out, p=self.dropout_p, training=self.training)
        x = self.norm2(x)

        ffn_out = self.ffn(x)
        x = x + F.dropout(ffn_out, p=self.dropout_p, training=self.training)
        x = self.norm3(x)

        return x

In [13]:
class Decoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_heads: int, d_ff: int,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.dropout_p = dropout_p

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout_p=dropout_p)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tgt_tokens, memory, tgt_attn_mask=None, src_key_padding_mask=None):
        x = self.tok_emb(tgt_tokens)
        x = self.pos_emb(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        for layer in self.layers:
            x = layer(x, memory, tgt_attn_mask=tgt_attn_mask, src_key_padding_mask=src_key_padding_mask)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [14]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int,
                 d_model: int = 128, num_layers: int = 2,
                 num_heads: int = 4, d_ff: int = 256,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, dropout_p, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout_p, max_len)

    def forward(self, src_tokens, tgt_tokens, src_pad_mask=None, tgt_causal_mask=None):
        memory = self.encoder(
            src_tokens,
            src_key_padding_mask=src_pad_mask
        )
        
        logits = self.decoder(
            tgt_tokens,
            memory,
            tgt_attn_mask=tgt_causal_mask,
            src_key_padding_mask=src_pad_mask
        )
        return logits

In [15]:
class CharTokenizer:
    def __init__(self, texts, pad_token="<pad>", bos_token="<bos>", eos_token="<eos>"):
        chars = set("".join(texts))
        self.pad_token = pad_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        self.itos = [pad_token, bos_token, eos_token] + sorted(chars)
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}

        self.pad_id = self.stoi[pad_token]
        self.bos_id = self.stoi[bos_token]
        self.eos_id = self.stoi[eos_token]

    def encode(self, s: str, add_special=True):
        ids = [self.stoi[c] for c in s]
        if add_special:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            tok = self.itos[i]
            if tok in (self.pad_token, self.bos_token, self.eos_token):
                continue
            out.append(tok)
        return "".join(out)

In [16]:
def pad_to_max(batch_ids, pad_id):
    max_len = max(len(x) for x in batch_ids)
    out = []
    for x in batch_ids:
        out.append(x + [pad_id] * (max_len - len(x)))
    return torch.tensor(out, dtype=torch.long)

@torch.no_grad()
def next_token_distribution(model, src_tokenizer, tgt_tokenizer, src_text: str, tgt_prefix: str):
    model.eval()

    src = torch.tensor([src_tokenizer.encode(src_text)], device=device)
    tgt = torch.tensor([tgt_tokenizer.encode(tgt_prefix)], device=device)

    src_pad = make_padding_mask(src, src_tokenizer.pad_id)
    Tt = tgt.size(1)
    causal = make_causal_mask(Tt, device=device)

    logits = model(src, tgt, src_pad_mask=src_pad, tgt_causal_mask=causal)
    last_logits = logits[:, -1, :]
    probs = torch.softmax(last_logits, dim=-1)[0]
    return probs

def train_step(model, optimizer, loss_fn, scaler, src, tgt, src_pad_id, tgt_pad_id):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]

    src_pad_mask = make_padding_mask(src, src_pad_id)
    tgt_pad_mask = make_padding_mask(tgt_in, tgt_pad_id)
    tgt_causal_mask = make_causal_mask(tgt_in.size(1), device=src.device)
    tgt_mask = tgt_pad_mask * tgt_causal_mask

    with autocast(enabled=torch.cuda.is_available()):
        logits = model(
            src,
            tgt_in,
            src_pad_mask=src_pad_mask,
            tgt_causal_mask=tgt_mask
        )
        B, T, V = logits.shape
        loss = loss_fn(logits.reshape(B * T, V), tgt_out.reshape(B * T))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    return loss.item(), model

In [17]:
def run_train(model, train_loader, epochs, src_tok, tgt_tok, checkpoint_path="prac3_checkpoint.pt"):
    scaler = GradScaler(enabled=torch.cuda.is_available())
    model.train()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_tok.pad_id)

    loss = 0
    start_iter = 1

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)

        same_src_vocab = checkpoint.get("src_vocab_size") == model.encoder.tok_emb.num_embeddings
        same_tgt_vocab = checkpoint.get("tgt_vocab_size") == model.decoder.tok_emb.num_embeddings

        if same_src_vocab and same_tgt_vocab:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_iter = checkpoint["iteration"] + 1
        else:
            print("Checkpoint пропущен: размер словаря не совпадает.")

    progress_bar = tqdm(range(start_iter, epochs + 1), desc=f"Processing")
    for epoch in progress_bar:
        num_batch = 0

        for src, tgt in train_loader:
            num_batch += 1

            src = src.to(device)
            tgt = tgt.to(device)

            loss, model = train_step(model, optimizer, loss_fn, scaler, src, tgt, src_tok.pad_id, tgt_tok.pad_id)

            progress_bar.set_postfix({
                "batch": f"{num_batch}/{len(train_loader)}",
                "loss": f"{loss:.4f}",
            })

        if epoch % 5 == 0:
            torch.save({
                "iteration": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss,
                "src_vocab_size": model.encoder.tok_emb.num_embeddings,
                "tgt_vocab_size": model.decoder.tok_emb.num_embeddings,
            }, checkpoint_path)


        
    return model

In [18]:
dataset = load_dataset("opus_books", "en-ru")

full_train = dataset["train"]

split = full_train.train_test_split(test_size=0.1, seed=42)

train_raw = split["train"]
valid_raw = split["test"]

train_pairs = [
    (x["translation"]["en"].strip(), x["translation"]["ru"].strip())
    for x in train_raw
]

valid_pairs = [
    (x["translation"]["en"].strip(), x["translation"]["ru"].strip())
    for x in valid_raw
]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/opus_books/resolve/1f9f6191d0e91a3c539c2595e2fe48fc1420de9b/opus_books.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since opus_books couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en-ru' at C:\Users\semen\.cache\huggingface\datasets\opus_books\en-ru\0.0.0\1f9f6191d0e91a3c539c2595e2fe48fc1420de9b (last modified on Sun Mar 15 15:53:50 2026).


In [19]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_tok, tgt_tok, max_src_len=128, max_tgt_len=128):
        self.pairs = pairs
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]

        src_ids = self.src_tok.encode(src_text)[:self.max_src_len]
        tgt_ids = self.tgt_tok.encode(tgt_text)[:self.max_tgt_len]

        return src_ids, tgt_ids

def make_collate_fn(src_pad_id, tgt_pad_id):
    def collate_fn(batch):
        src_batch, tgt_batch = zip(*batch)
        src = pad_to_max(src_batch, src_pad_id)
        tgt = pad_to_max(tgt_batch, tgt_pad_id)
        return src, tgt
    return collate_fn

In [20]:
train_src_texts = [src for src, _ in train_pairs]
train_tgt_texts = [tgt for _, tgt in train_pairs]

src_tok = CharTokenizer(train_src_texts)
tgt_tok = CharTokenizer(train_tgt_texts)

train_ds = TranslationDataset(train_pairs[:30000], src_tok, tgt_tok, max_src_len=128, max_tgt_len=128)
valid_ds = TranslationDataset(valid_pairs[:2000], src_tok, tgt_tok, max_src_len=128, max_tgt_len=128)

collate_fn = make_collate_fn(src_tok.pad_id, tgt_tok.pad_id)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [21]:
model = TransformerSeq2Seq(
    src_vocab_size=len(src_tok.itos),
    tgt_vocab_size=len(tgt_tok.itos),
    d_model=256,
    num_layers=4,
    num_heads=8,
    d_ff=1024,
    dropout_p=0.1,
).to(device)

In [22]:
model = run_train(model= model, train_loader= train_loader, epochs= 30, src_tok= src_tok, tgt_tok= tgt_tok)

C:\Users\semen\AppData\Local\Temp\ipykernel_5320\1288460685.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
C:\Users\semen\AppData\Local\Temp\ipykernel_5320\1288460685.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you s

In [23]:
model.eval()

src_text, tgt_text = valid_pairs[1]
prefix = tgt_text[:4]

with torch.no_grad():
    probs = next_token_distribution(model, src_tok, tgt_tok, src_text, prefix)
    topk = torch.topk(probs, k=10)

print("SRC:", src_text)
print("TRUE TGT:", tgt_text)
print("TGT prefix:", repr(prefix))
for p, idx in zip(topk.values.tolist(), topk.indices.tolist()):
    print(f"token={repr(tgt_tok.itos[idx])} prob={p:.4f}")

SRC: While they were talking Laska, pricking her ears, kept looking up at the sky and then reproachfully at them.
TRUE TGT: В то время, как они говорили это, Ласка, насторожив уши, оглядывалась вверх на небо и укоризненно на них.
TGT prefix: 'В то'
token='т' prob=0.8709
token=' ' prob=0.0801
token='и' prob=0.0102
token='л' prob=0.0080
token='к' prob=0.0070
token='ш' prob=0.0051
token='я' prob=0.0037
token=',' prob=0.0031
token='н' prob=0.0021
token='р' prob=0.0018


SRC: While they were talking Laska, pricking her ears, kept looking up at the sky and then reproachfully at them.
TRUE TGT: В то время, как они говорили это, Ласка, насторожив уши, оглядывалась вверх на небо и укоризненно на них.
token='д' prob=0.2834
token='-' prob=0.1088
token='у' prob=0.0953
token='к' prob=0.0887
token='а' prob=0.0612
token='р' prob=0.0560
token='и' prob=0.0313
token='В' prob=0.0256
token='о' prob=0.0247
token='О' prob=0.0192

token='-' prob=0.3457
token='д' prob=0.2724
token='р' prob=0.1317
token='о' prob=0.0687
token=' ' prob=0.0502
token='в' prob=0.0190
token='к' prob=0.0168
token='у' prob=0.0130
token='с' prob=0.0128
token='т' prob=0.0128

token='а' prob=0.1498
token='р' prob=0.1316
token='д' prob=0.1211
token='в' prob=0.0843
token='г' prob=0.0783
token='о' prob=0.0678
token='л' prob=0.0641
token='н' prob=0.0565
token='м' prob=0.0467
token='у' prob=0.0396

token='ж' prob=0.1859
token=' ' prob=0.1571
token='и' prob=0.1406
token='н' prob=0.0809
token='о' prob=0.07